In [1]:
import numpy as np
import pandas as pd
from scipy.signal import welch

FS      = 200
WINDOW  = 200   #1 second
STEP    = 100   #50% overlap between windows



In [ ]:
#Feature functions

def compute_rms(signal):
    return np.sqrt(np.mean(signal ** 2))

def compute_mav(signal):
    # Mean Absolute Value — mean of rectified signal
    return np.mean(np.abs(signal))

def compute_wl(signal):
    # Waveform Length — sum of absolute differences
    return np.sum(np.abs(np.diff(signal)))

def compute_zcr(signal):
    # Zero Crossing Rate — how often signal crosses zero
    return np.sum(np.diff(np.sign(signal)) != 0)

def compute_mdf(signal, fs):
    f, Pxx   = welch(signal, fs=fs, nperseg=len(signal))
    cumsum   = np.cumsum(Pxx)
    return f[np.where(cumsum >= cumsum[-1] / 2)[0][0]]


def extract_features(raw, env, fs):
    return {
        "rms":  compute_rms(env),
        "mav":  compute_mav(env),
        "wl":   compute_wl(raw),
        "zcr":  compute_zcr(raw),
        "mdf":  compute_mdf(raw, fs),
    }


# --- Process a CSV file and return a feature dataframe ---
def process_file(csv_path, label, fs=FS, window=WINDOW, step=STEP):
    df      = pd.read_csv(csv_path)
    raw_all = df["Raw_EMG"].values
    env_all = df["Envelope_EMG"].values

    rows = []
    for start in range(0, len(raw_all) - window, step):
        end = start + window
        raw = raw_all[start:end]
        env = env_all[start:end]
        features        = extract_features(raw, env, fs)
        features["label"] = label
        rows.append(features)

    return pd.DataFrame(rows)


In [3]:
# --- Build dataset from multiple files ---
# Label 0 = non-fatigued, Label 1 = fatigued
# Record separate sessions or split a single session manually

files = [
    # (csv_path,                              label)
    ("D:\\Project\\Dataset EMG Fatigue\\candy_read\\data\\Athul\\session1_nonfatigued.csv", 0),
    ("D:\\Project\\Dataset EMG Fatigue\\candy_read\\data\\Athul\\session1_fatigued.csv",    1),
]



In [ ]:
all_frames = [process_file(path, label) for path, label in files]
dataset    = pd.concat(all_frames, ignore_index=True)
dataset.to_csv("D:\\Project\\Dataset EMG Fatigue\\candy_read\\data\\Athul\\features_dataset_BL.csv", index=False)


label
0    42
1    25
Name: count, dtype: int64
          rms     mav       wl  zcr   mdf  label
0  120.477052  120.38  16605.0   99  51.0      0
1  114.142893  113.88  15659.0  100  51.0      0
2  112.660197  112.44  15331.0   93  46.0      0
3  126.489841  126.04  17828.0   87  45.0      0
4  136.894485  136.71  18324.0   90  44.0      0


In [7]:
print(dataset["label"].value_counts())
print(dataset.head(5))

label
0    42
1    25
Name: count, dtype: int64
          rms     mav       wl  zcr   mdf  label
0  120.477052  120.38  16605.0   99  51.0      0
1  114.142893  113.88  15659.0  100  51.0      0
2  112.660197  112.44  15331.0   93  46.0      0
3  126.489841  126.04  17828.0   87  45.0      0
4  136.894485  136.71  18324.0   90  44.0      0
